# 08 — Complete RAG: From Question to Grounded Answer

We have a retrieval pipeline. We have a model. Today we connect them, and StoryVerse AI stops guessing.

```
[ Retrieval Pipeline ] -> Retrieved Chunks
                                |
                                v
                        [ Prompt Builder ]
                                |
                                v
                           [ Groq Model ]
                                |
                                v
                  Grounded Answer + Sources
```


In [ ]:
import os
import json
import faiss
import numpy as np
from dotenv import load_dotenv
from groq import Groq
from sentence_transformers import SentenceTransformer

load_dotenv()

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-20b"
embedder = SentenceTransformer("all-MiniLM-L6-v2")

DATA_DIR = os.path.join("..", "data")
faiss_chunks_path = os.path.join(DATA_DIR, "storyverse_chunks.faiss")
doc_store_path = os.path.join(DATA_DIR, "storyverse_chunks_docstore.json")

index = faiss.read_index(faiss_chunks_path)
with open(doc_store_path) as f:
    doc_store = json.load(f)

print(f"Loaded index from notebook 07: {index.ntotal} chunks")


def retrieve_with_threshold(query: str, index, doc_store, top_k: int = 3, min_score: float = 0.35) -> list[dict]:
    query_vec = embedder.encode([query]).astype(np.float32)
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, top_k)
    results = [{**doc_store[i], "score": float(scores[0][j])} for j, i in enumerate(indices[0])]
    return [r for r in results if r["score"] >= min_score]


# sanity check
print(retrieve_with_threshold("what happens in the wormhole", index, doc_store, top_k=1))


## The most important function in this whole notebook

Everything about whether RAG "works" or "hallucinates anyway" comes down to how this prompt is built.


In [ ]:
def build_rag_prompt(question: str, retrieved_chunks: list[dict]) -> str:
    context_parts = [
        f"[Source: {c['metadata']['source']}]\n{c['content']}"
        for c in retrieved_chunks
    ]
    context = "\n\n".join(context_parts)

    return f"""You are StoryVerse AI, an assistant that answers questions about movies and stories.

Answer the question using ONLY the context provided below. If the answer is not
in the context, say exactly: "I don't have enough information about this in my
knowledge base." Do not guess.

Context:
{context}

Question: {question}

Answer:"""


Two design choices are load-bearing here: **"using ONLY the context"** stops the model from falling back on its own parametric memory (the exact failure from notebook 00), and the explicit fallback instruction gives it a safe, honest way to say it doesn't know instead of guessing anyway.


## Assembling `rag_answer`


In [ ]:
def rag_answer(question: str, index, doc_store, top_k: int = 3, min_score: float = 0.35) -> dict:
    chunks = retrieve_with_threshold(question, index, doc_store, top_k, min_score)

    if not chunks:
        return {
            "answer": "I don't have enough information about this in my knowledge base.",
            "sources": [],
            "retrieved_chunks": 0,
        }

    prompt = build_rag_prompt(question, chunks)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
    )

    return {
        "answer": response.choices[0].message.content,
        "sources": sorted(set(c["metadata"]["source"] for c in chunks)),
        "retrieved_chunks": len(chunks),
        "chunks_used": chunks,
    }


result = rag_answer("Who is Cooper's daughter in Interstellar?", index, doc_store)
print("Answer: ", result["answer"])
print("Sources:", result["sources"])


## The big demo: with RAG vs. without

Same questions, bare model vs. grounded model, side by side. This is the whole series distilled into one cell.


In [ ]:
def ask_bare(question: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=0.2,
    )
    return response.choices[0].message.content


demo_questions = [
    "What is the name of Cooper's daughter in Interstellar?",
    "What happens to Arjun in the bookshop story?",
    "Who does Shivudu fall in love with in Baahubali?",
    "Why does Batman take the blame for Harvey Dent's crimes?",
    "What happens in the ending of 'The Crimson Algorithm'?",  # doesn't exist -- deliberate trap
]

for q in demo_questions:
    bare = ask_bare(q)
    rag = rag_answer(q, index, doc_store)

    print("=" * 60)
    print("Question:", q)
    print("\nWithout RAG:\n ", bare)
    print("\nWith RAG:\n ", rag["answer"], "\n  Sources:", rag["sources"])
    print()


The last question is a trap on purpose — "The Crimson Algorithm" is the fictional movie from notebook 00 and it isn't in our knowledge base at all. Watch what each version does with it: the bare model likely fabricates a plot again, while the RAG version should honestly say it doesn't know, because nothing in the retrieved chunks matches and the threshold filters it out.


## Source attribution

Users trust an answer more when they can see where it came from. We're already tracking this in `rag_answer` — let's format it properly.


In [ ]:
result = rag_answer("Who are Harry's two best friends?", index, doc_store)
print("Answer: ", result["answer"])
print(f"Based on: {', '.join(result['sources'])}")
for c in result["chunks_used"]:
    print(f"  - {c['metadata']['source']} (chunk #{c['metadata']['chunk_index']}, score {c['score']:.2f})")


## The "I don't know" behavior, tested directly

Oppenheimer isn't in our library at all.


In [ ]:
result = rag_answer("What happens in Oppenheimer?", index, doc_store)
print("With threshold filtering:", result["answer"])

# now WITHOUT the threshold, to see the difference
unfiltered_chunks = retrieve_with_threshold("What happens in Oppenheimer?", index, doc_store, top_k=3, min_score=0.0)
prompt = build_rag_prompt("What happens in Oppenheimer?", unfiltered_chunks)
response = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0.1)
print("\nWithout threshold filtering (low-confidence chunks still passed in):", response.choices[0].message.content)


Teaching a model to admit "I don't know" is one of the harder parts of production RAG — instructions alone are a soft guardrail, and a model can still be talked into guessing if you feed it weak, barely-related context. A score threshold is a hard guardrail: no relevant chunks, no attempt at an answer.


## The same system, the LangChain way


In [ ]:
from langchain_classic.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

lc_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
llm = ChatGroq(model=MODEL, temperature=0.1)

chroma_store = Chroma.from_texts(
    texts=[d["content"] for d in doc_store],
    embedding=lc_embeddings,
    metadatas=[d["metadata"] for d in doc_store],
    collection_name="storyverse_rag",
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=chroma_store.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
)

lc_result = qa_chain.invoke({"query": "Who is Cooper's daughter in Interstellar?"})
print("LangChain answer:", lc_result["result"])
print("Source chunks used:", [d.metadata for d in lc_result["source_documents"]])


`RetrievalQA.from_chain_type` is doing exactly what `rag_answer` does by hand: retrieve, build a prompt, call the model, return sources. Having built it ourselves first is what makes this one-liner make sense instead of feeling like magic.


## Does RAG actually reduce hallucination? Let's grade it

Five questions, both versions, manually labeled.


In [ ]:
test_cases = [
    {"question": "Who is Cooper's daughter in Interstellar?", "expected": "Murph"},
    {"question": "What object is Voldemort trying to steal?", "expected": "The Sorcerer's Stone"},
    {"question": "Who does Shivudu fall in love with?", "expected": "Avanthika"},
    {"question": "Why does Batman take the blame for Harvey Dent's crimes?", "expected": "to protect Gotham's faith in its 'white knight'"},
    {"question": "What happens in 'The Crimson Algorithm'?", "expected": "not in our knowledge base -- should decline to answer"},
]

print(f"{'Question':<55}{'Bare model':<15}{'RAG':<15}")
for case in test_cases:
    bare = ask_bare(case["question"])
    rag = rag_answer(case["question"], index, doc_store)
    # Labeling is manual/eyeballed here -- read the printed answers and judge for yourself
    print(f"{case['question'][:53]:<55}{'(see below)':<15}{'(see below)':<15}")
    print(f"   expected: {case['expected']}")
    print(f"   bare    : {bare}")
    print(f"   rag     : {rag['answer']}")
    print()


RAG doesn't make hallucination impossible — a model can still misread retrieved text. What it does is remove the single biggest cause of it: answering from memory when the real answer was never in memory to begin with.


## What actually made the grading go well

Every "correct" answer above traces back to a handful of small choices in `build_rag_prompt`, not luck. Worth writing down, since it's the part people get wrong when they build their own:

- Always say **"answer ONLY using the provided context"** — this is the single highest-leverage sentence in the whole system
- Always give the model an explicit, exact fallback phrase for missing information
- Label sources in the context, so the model (and you) can cite them
- Keep instructions short — a bloated system prompt competes with the context for attention
- Use low temperature (0.0-0.1) for factual answers

Common mistakes that undo all of this: skipping the fallback instruction (the model fills the gap with a guess), cranking temperature up (the model starts "interpreting" context loosely instead of reading it), and stuffing in too many chunks (the context dilution problem from notebook 03 comes right back).


## A real, interactive StoryVerse AI

Run the cell below, and actually talk to it.


In [ ]:
print("StoryVerse AI -- ask me about movies and stories!")
print("Type 'quit' to exit.\n")

while True:
    question = input("You: ").strip()
    if question.lower() == "quit":
        break
    result = rag_answer(question, index, doc_store)
    print(f"\nStoryVerse AI: {result['answer']}")
    if result["sources"]:
        print(f"Sources: {', '.join(result['sources'])}\n")
    else:
        print()


Not conversational yet — no memory of earlier turns in this chat — but it retrieves and answers correctly, with sources. That's a real RAG system.


## What StoryVerse AI can actually do now

Talk to it yourself in the cell above, then look at what just answered you:

- **Retrieval** — find the right chunks (notebook 07)
- **Grounding** — force the model to answer only from those chunks
- **Source attribution** — show exactly which chunk backs the answer
- **Graceful refusal** — say "I don't know" instead of guessing, when nothing relevant is found

Still missing: what happens when retrieval itself goes wrong (next notebook), conversational memory, and a way to measure quality with numbers instead of eyeballing it (notebook 10).

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Grounding | Forcing an answer to be based on retrieved text, not the model's memory |
| Source attribution | Showing which document/chunk an answer came from |
| Fallback instruction | An explicit, safe phrase the model should use when it truly doesn't know |

**Exercises:**

- Add 3 new story files, re-run notebook 07's indexing, and confirm they're answerable here.
- Try different `top_k` values in `rag_answer` — how does the answer's detail change?
- Remove the "using ONLY the context" line from `build_rag_prompt` and re-run the Crimson Algorithm question — what happens?
